# Radial Basis Nearest Neighbor (RBNN)

- Giulia Monteiro Garrido (RA: 24010281)
- Mateus Antezana da Silva (RA: )
- Vitor Furuta da Silva (RA: 24008775)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Sequence

In [3]:
class Distancias:
    def __init__(self):
        pass

    def _validar_vetores(self, A:Sequence[int | float], B:Sequence[int | float]):
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)

        if A.shape != B.shape:
            raise ValueError("Os vetores devem ter a mesma dimensão.")

        if A.size == 0:
            raise ValueError("Os vetores não podem estar vazios.")

        return A, B

    def distancia(self, funcao, A:Sequence[int | float], B:Sequence[int | float]) -> int | float | None:

        try:
            A, B = self._validar_vetores(A, B)
            return funcao(A, B)
        
        except IndexError:
            print(IndexError)
            
        except ValueError:
            print(ValueError)

    def minkowski(self, A:Sequence[int | float], B:Sequence[int | float], p:int) -> float | int | None:

        try: 
            
            if p <= 0:
                raise ValueError("p deve ser maior que zero.")
            
            A, B = self._validar_vetores(A, B)

            return sum(abs(A[i]-B[i])**p for i in range(len(A))) ** (1/p)

        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
        

    def cosseno(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:
        
        try:
            A, B = self._validar_vetores(A, B)

            produto = np.dot(A, B)
            norma_A = np.linalg.norm(A)
            norma_B = np.linalg.norm(B)

            if norma_A == 0 or norma_B == 0:
                return 0

            return float(1 - (produto/ (norma_A * norma_B)))
        
        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
        
    def manhattan(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:

        try:
            A,B = self._validar_vetores(A,B)
            return sum((abs(A[i]-B[i])) for i in range(len(A)))

    
        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)
    
    def euclidiana(self, A:Sequence[int | float], B:Sequence[int | float]) -> float | int | None:

        try:
            A,B = self._validar_vetores(A,B)
            return sum(((A[i]-B[i])**2) for i in range(len(A)))**(1/2)

        except IndexError:
            print(IndexError)
                    
        except ValueError:
            print(ValueError)

NameError: name 'Sequence' is not defined

In [4]:
def pesos(x, X_treino, sigma, distancia):

    distancias = np.array([
        distancia(x, xi) for xi in X_treino
    ])
    formula = 1 / (sigma * np.sqrt(2 * np.pi))
    pesos = formula * np.exp(-(distancias ** 2) / (2 * sigma ** 2))
    return pesos


def regressao(x, X_treino, y_treino, sigma, distancia):

    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    return np.sum(w * y_treino) / np.sum(w)

def classificacao(x, X_treino, y_treino, sigma, distancia):

    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    y_treino = np.asarray(y_treino)
    classes = np.unique(y_treino)
    votos = {c: w[y_treino == c].sum() for c in classes}
    return max(votos, key=votos.get)


In [5]:
class RBNNRegressor:
    def __init__(self, sigma: float, distancia):
        self.sigma = sigma
        self.distancia = distancia

    def fit(self, X_treino, y_treino):
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    def predict(self, X_teste):
        return np.array([
            regressao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

In [6]:
class RBNNClassifier:
    def __init__(self, sigma: float, distancia):
        self.sigma = sigma
        self.distancia = distancia

    def fit(self, X_treino, y_treino):
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    def predict(self, X_teste):
        return np.array([
            classificacao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

## Carregando bases

In [ ]:
iris = sns.load_dataset("iris")

X_iris = iris.drop(columns="species").to_numpy()
y_iris = iris["species"].to_numpy()

print(iris.shape)          # (150, 5)
iris.head()
